# EDA — NS Health Atlas: Community Environs
**Objetivo:** Entender los 238 indicadores disponibles por CE, identificar calidad de datos y seleccionar los más relevantes para el perfil comunitario del paciente en HealthLocate.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

## 1. Carga y estructura del dataset

In [ ]:
df = pd.read_csv(r"C:\Users\maria\OneDrive\Escritorio\HealthLocate\data\NSHealthAtlasDataEnvirons.csv")
df_clean = df[df['region'] == 'community-environs'].copy()

print(f"Community Environs (CEs): {len(df_clean)}")
print(f"Indicadores (columnas totales): {len(df_clean.columns)}")
df_clean.head(3)

## 2. Grupos de indicadores

Los 238 campos se organizan en estos grupos temáticos:

| Prefijo | Tema | # cols aprox. |
|---------|------|---------|
| `pctpop` / `pop` | Demografía / estructura etaria | ~54 |
| `medage` | Edad mediana | 3 |
| `msi-` | Material & Social Index (privación material) | 8 |
| `scs-` | Social Conditions Score | 4 |
| `sds-` | Social Diversity Score | 5 |
| `foc-` | Indicadores Focused on Communities (census) | 11 |
| `green-` | Vegetación / espacio verde (NDVI) | 1 |
| `aq-` | Calidad del aire (PM2.5, NOx, NO2, radón) | 10 |
| `well-` | Calidad del agua de pozos privados | 4 |
| `smk-` | Tabaquismo (curr/fmr/shs por sexo) | 24 |
| `cancer-` | Incidencia de cánceres por tipo y sexo | ~115 |

In [ ]:
prefixes = ['pctpop','pop','medage','msi-','scs-','sds-','foc-','green-','aq-','well-','smk-','cancer-']
for p in prefixes:
    cols = [c for c in df_clean.columns if c.startswith(p)]
    print(f"{p:<12} → {len(cols):>3} cols")

## 3. Calidad de datos — valores faltantes

In [ ]:
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(1)
missing_df = pd.DataFrame({'missing_n': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_n'] > 0].sort_values('missing_pct', ascending=False)

print(f"Columnas con datos faltantes: {len(missing_df)} de {len(df_clean.columns)}")
print()
print(missing_df.to_string())

In [ ]:
# Solo columnas con nulos
null_cols = missing_df.index.tolist()
msno.bar(df_clean[null_cols], figsize=(6, 3), fontsize=11, color='steelblue')
plt.title('Completitud — columnas con valores faltantes')
plt.tight_layout()
plt.show()

print("\nNota: well-arsenic, well-uranium, well-manganese tienen ~29% nulos")
print("Corresponde a CEs sin pozos privados — ausencia informativa, no error de datos.")

## 4. Distribuciones — indicadores socioeconómicos y ambientales clave

In [ ]:
socio_cols = {
    'msi-score2021': 'Material & Social Index (1-5)',
    'msi-lowincome': 'Bajo ingreso (% hogares)',
    'msi-unemploymentrate': 'Tasa de desempleo',
    'foc-licoaftax': 'LIM-AT (% bajo línea pobreza)',
    'foc-emprate': 'Tasa de empleo',
    'foc-subhous': 'Vivienda inadecuada (% hogares)',
    'foc-renters': 'Arrendatarios (% hogares)',
    'medage_total': 'Edad mediana'
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, (col, label) in zip(axes.flat, socio_cols.items()):
    sns.histplot(df_clean[col].dropna(), ax=ax, bins=20, kde=True, color='steelblue')
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Distribución — Indicadores Socioeconómicos por CE', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
env_cols = {
    'green-pwndvi': 'Vegetación / NDVI (0-1)',
    'aq-meanpm25': 'PM2.5 promedio (μg/m³)',
    'aq-radon': 'Radón (categoría)',
    'well-arsenic': 'Arsénico en agua (% pozos >límite)',
    'smk-curr_male': 'Fumadores actuales — hombres',
    'smk-curr_female': 'Fumadores actuales — mujeres'
}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, (col, label) in zip(axes.flat, env_cols.items()):
    sns.histplot(df_clean[col].dropna(), ax=ax, bins=20, kde=True, color='seagreen')
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Distribución — Indicadores Ambientales y Comportamentales por CE', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. Mapa de calor — correlaciones entre indicadores seleccionados

In [ ]:
profile_cols = [
    'pop_total_all', 'medage_total', 'pctpop-pct_total_65',
    'msi-score2021', 'msi-lowincome', 'msi-unemploymentrate',
    'scs-score2021', 'sds-score2021',
    'foc-licoaftax', 'foc-emprate', 'foc-subhous', 'foc-renters',
    'green-pwndvi', 'aq-meanpm25', 'aq-radon',
    'smk-curr_male', 'smk-curr_female'
]

corr = df_clean[profile_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.3, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Correlación entre indicadores del perfil comunitario', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Variabilidad entre CEs — ¿qué indicadores diferencian más las comunidades?

In [ ]:
# Coeficiente de variación (CV) = std / mean — mide cuánto varía cada indicador entre CEs
cv = (df_clean[profile_cols].std() / df_clean[profile_cols].mean()).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
cv.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Coeficiente de Variación (std / media)')
ax.set_title('Variabilidad entre CEs por indicador (CV)')
ax.axvline(0.3, color='red', linestyle='--', linewidth=0.8, label='CV = 0.30')
ax.legend()
plt.tight_layout()
plt.show()

print("\nCV por indicador:")
print(cv.round(3).to_string())

## 7. Distribución del MSI por quintil — ancla del perfil de privación

In [ ]:
msi_counts = df_clean['msi-score2021'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2ecc71','#a9dfbf','#f9e79f','#f0b27a','#e74c3c']
ax.bar(msi_counts.index, msi_counts.values, color=colors, edgecolor='white')
ax.set_xlabel('Quintil MSI (1 = menos privado, 5 = más privado)')
ax.set_ylabel('Número de CEs')
ax.set_title('Distribución de CEs por quintil MSI 2021')
for i, (idx, val) in enumerate(msi_counts.items()):
    ax.text(idx, val + 1, str(val), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Indicadores seleccionados para el perfil del paciente

Basado en el EDA, estos son los indicadores más relevantes y de mejor calidad para el **perfil comunitario** en HealthLocate:

In [ ]:
shortlist = {
    # --- Demografía ---
    'pop_total_all':          'Población total',
    'medage_total':           'Edad mediana',
    'pctpop-pct_total_65':    '% población ≥65 años',

    # --- Privación material y social ---
    'msi-score2021':          'Índice MSI 2021 (quintil 1-5)',
    'msi-lowincome':          '% hogares bajo ingreso',
    'msi-unemploymentrate':   'Tasa de desempleo',
    'foc-licoaftax':          '% bajo línea de pobreza (LIM-AT)',

    # --- Condiciones de vivienda ---
    'foc-subhous':            '% vivienda inadecuada',
    'foc-renters':            '% arrendatarios',

    # --- Condiciones sociales y diversidad ---
    'scs-score2021':          'Score condiciones sociales (quintil)',
    'scs-loneparent':         '% familias monoparentales',
    'sds-score2021':          'Score diversidad social (quintil)',

    # --- Ambiente físico ---
    'green-pwndvi':           'Cobertura vegetal (NDVI)',
    'aq-meanpm25':            'PM2.5 promedio (μg/m³)',
    'aq-radon':               'Nivel de radón (categoría)',

    # --- Comportamental ---
    'smk-curr_male':          'Prevalencia tabaquismo — hombres',
    'smk-curr_female':        'Prevalencia tabaquismo — mujeres',
}

sl_df = pd.DataFrame.from_dict(shortlist, orient='index', columns=['Descripción'])
sl_df.index.name = 'Campo'

# Stats de cada indicador seleccionado
sl_df['Media'] = df_clean[sl_df.index].mean().round(3)
sl_df['Std']   = df_clean[sl_df.index].std().round(3)
sl_df['Nulos %'] = (df_clean[sl_df.index].isnull().mean() * 100).round(1)

sl_df

## 9. Ejemplo: perfil de una CE específica

In [ ]:
# Ver perfil de una CE de ejemplo
ce_name = 'Halifax'  # cambiar por cualquier CE

match = df_clean[df_clean['name'].str.contains(ce_name, case=False, na=False)]
if match.empty:
    print(f"CE '{ce_name}' no encontrada. CEs disponibles (muestra):")
    print(df_clean['name'].sample(10).tolist())
else:
    row = match.iloc[0]
    print(f"CE: {row['name']} (id={int(row['id'])})\n")
    for field, desc in shortlist.items():
        val = row.get(field, None)
        print(f"  {desc:<45} {val}")

## Conclusiones del EDA

### Estructura del dataset
- **300 Community Environs** con **238 columnas** — excelente cobertura geográfica de toda Nova Scotia.
- Solo **3 columnas** tienen datos faltantes (`well-arsenic`, `well-uranium`, `well-manganese`, ~29% nulos), y corresponden a CEs sin pozos privados — ausencia informativa, no un error.

### Indicadores con mayor poder diferenciador entre CEs
Los indicadores con mayor CV (variabilidad relativa) son:
- Indicadores de pobreza/bajo ingreso (`foc-licoaftax`, `msi-lowincome`)
- Vivienda inadecuada y arrendatarios
- Diversidad social y familias monoparentales
- NDVI (cobertura vegetal)

### Indicadores menos útiles para el perfil
- Las columnas de cáncer por tipo tienen baja variabilidad entre CEs y alta granularidad redundante (lc/uc/exceed por sexo × tipo → ~115 cols). Para el perfil del paciente no son prácticas en esta fase.
- `pctpop-pct_total_all` es la fracción provincial, no útil como perfil individual.

### Shortlist recomendada: 17 indicadores
Ver celda 8 — cubre demografía, privación, vivienda, diversidad social, ambiente físico y tabaquismo. Todos con completitud ≥100% (excepto pozos, que son opcionales).

### Próximos pasos
- **Tarea 4:** Vincular la shortlist con el motor geográfico (address → CE id) para construir el perfil comunitario del paciente.